# Initiation à EOReader

## EOReader

### Généralités

[EOReader](https://eoreader.readthedocs.io/stable/) est une bibliothèque Python open source.  
Elle permet de lire des images satellite optiques et SAR de façon agnostique afin d'abstraire les spécificités de chaque capteur.  
EOReader permet en outre de calculer des indices spectraux, des bandes associées au MNT (pente, hillshade...) ainsi que de charger les masques (nuages ou autres).

EOReader est initialement une brique des algorithmes [ICube-Sertit](https://sertit.unistra.fr/) dédiés à la cartographie rapide, notamment [Copernicus Emergency Mapping](https://mapping.emergency.copernicus.eu/activations/).  
En crise, il est impératif de supporter un maximum de capteurs différents pour maximiser la fréquence de revisite d'une zone.  
EOReader permet de créer des algorithmes indépendants du type de capteur en entrée.

Pour aller plus loin, ce [notebook](https://eoreader.readthedocs.io/latest/notebooks/why_eoreader.html) vous permettra de comprendre l'abstraction d'EOReader comparé à la lecture des données via `rasterio`.

<img src="https://zupimages.net/up/22/27/6yz0.png" alt="EOReader vs Rasterio" width="800">


### Capteurs optiques supportés

<img src="https://zupimages.net/up/26/04/xtdk.png" alt="Capteurs optiques supportés" width="800">

### Principe de correspondance des bandes spectrales

<a href="https://eoreader.readthedocs.io/latest/optical_band_mapping.html"><img src="https://zupimages.net/up/25/44/i20v.png" alt="Correspondances entres bandes spectrales" width="800"></a>

### Capteurs SAR supportés

<img src="https://i.postimg.cc/GhPpwwL7/sar-sensors.png" alt="Capteurs SAR supportés" width="800">

⚠️ Il est nécessaire d'avoir à disposition ESA SNAP pour gérer les capteurs SAR.

## Principe du notebook

### Déroulé
Nous allons chercher deux images de capteurs gratuits sur Microsoft Planetary Computer couvrant **Saumos** en Gironde, le feu catastrophique de cet été. 

Le but est d'essayer d'extraire rapidement une zone brulée basée sur deux capteurs (Sentinel-2, Landsat) grace à l'indice spectral NBR.  
Nous calculerons le NBR des images pré et post évènement puis nous seuillerons la différence de ces indices pour en extraire la tâche de feu. 

<img src="https://france3-regions.franceinfo.fr/image/ITZQVosBO01UoK5f9IwXEURDicI/3376x2251/regions/2026/07/29/6a69b39c52555364361840.jpg" alt="Feux Saumos" width="600">

Nous réitèrerons le procédé avec la bibliothèque EOSets, conçue pour gérer plusieurs produits EOReader, afin de voir comment son utilisation simplifie le code.

### Objectifs

- Prendre en main EOReader
- Comprendre les cas d'usages dans lesquels EOreader est pertinent
- Initiation à EOSets
- Tour d'horizon de l'écosystème sur lequel EOReader est basé

## Première partie : Extraction de feu basée sur EOReader

### Imports

In [ ]:
%pip install -r requirements.txt

In [ ]:
import warnings
import contextlib 
from rasterio.errors import NotGeoreferencedWarning
import geopandas as gpd
import matplotlib.pyplot as plt

from eoreader.reader import Reader
from eoreader.bands import MNDWI, NBR

In [ ]:
# --- Workaround pour Jupyter ---
# Faire fonctionner 'asyncio.run()' dans les notebooks Jupyter 
import nest_asyncio2
nest_asyncio2.apply()

# Accès aux données MPC depuis le cluster du CNES
import os
os.environ["GDAL_HTTP_UNSAFESSL"] = "YES"

In [ ]:
# Creation du logger d'EOReader
import logging
from sertit import logs  # Just this call to Sertit's internal lib to simplify logger creation

logs.init_logger(logging.getLogger("eoreader"), log_lvl=logging.DEBUG)
logs.init_logger(logging.getLogger("eosets"), log_lvl=logging.DEBUG)

# Filtrage des warnings non voulus
warnings.filterwarnings(
    "ignore",
    message=".*has no geotransform.*",
    category=NotGeoreferencedWarning,
)
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

### Données

Par soucis de simplicité, nous irons chercher nos données sur [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/). Cependant, EOReader a été principalement designé pour fonctionner avec des données sur disque.

Nous avons besoin d'une image pré-évènement et d'une image post-évènement. Nous allons ouvrir les deux chemins avec l'objet Reader d'EOReader.

In [ ]:
# Chemins Microsoft Planetary Computer
# En cas de besoin de trouver d'autres données, il peut être utile de prendre en main la bibliothèque EODAG.
s2_pre_path = "https://planetarycomputer.microsoft.com/api/stac/v1/collections/sentinel-2-l2a/items/S2C_MSIL2A_20260721T105621_R094_T30TXQ_20260721T144959"
l9_post_path = "https://planetarycomputer.microsoft.com/api/stac/v1/collections/landsat-c2-l2/items/LC09_L2SP_200029_20260807_02_T1"
aoi_path = "saumos.geojson"

In [ ]:
# Création du singloton du lecteur de produits satellite
reader = Reader()

# Lecture des données via EOReader
pre = reader.open(s2_pre_path)
post = reader.open(l9_post_path)

# Lecture de la zone d'intérêt
aoi = gpd.read_file(aoi_path)

Comme nous le voyons, il n'est pas utile de spécifier à quel capteur nous avons à faire : EOReader les reconnait automatiquement.

Voyons ce que contiennent ces produits satellite.

In [ ]:
# Représentation de l'image pré
pre

In [ ]:
# Bandes contenues dans l'image pré
pre.bands

In [ ]:
# Représentation de l'image post
post

EOReader vient avec un concept de nom unifié : le `condensed_name`. Ce nom reprend la même structure peut importe les capteurs et permet de facilement trier les fichiers.

Affichons ensuite les quicklooks puis vérifions que ces données se superposent bien. Pas de dNBR sans superposition.

In [ ]:
# Quicklook de l'image pré
pre.plot()

In [ ]:
# Quicklook de l'image post
post.plot()

In [ ]:
# Informations géographiques
print(f"CRS de l'image pré : EPSG:{pre.crs().to_epsg()}")
print(f"CRS de l'image post : EPSG:{post.crs().to_epsg()}")

⚠️ EOReader fonctionne toujours en UTM quelque soit le CRS initial. Une donnée en coordonnées géographiques sera reprojetées dans l'UTM s'approchant le plus.

In [ ]:
# Calcul des footprints des produits
pre_footprint = pre.footprint()
post_footprint = post.footprint()

# Vérification de la superposition des footprints et de la zone d'intéret
base = post_footprint.boundary.plot(edgecolor='red', label="Image post (Landsat-9)", tiles=True)
pre_footprint.boundary.plot(ax=base, edgecolor='green', label="Image pré (Sentinel-2)", tiles=True)
aoi.to_crs(pre_footprint.crs).boundary.plot(ax=base, edgecolor='blue', label="Zone d'intéret")
plt.legend()

### Extraction thématique

Il est maintenant temps de loader les indices spectraux `NBR` (Normalized Burnt Ratio) pré et post. 

Il est inutile de loader la bande Sentinel-2 avec une résolution de 10 mètre car la Landsat a une résolution de 30 mètres.  
Nous ne loaderons aussi que la zone couvrant l'AOI.

Nous loaderons aussi l'indice spectral `MNDWI` (Modified Normalized Difference Water Index) afin de nettoyer les zones en eau du résultat final.

💡EOReader se sert de la librairie [spyndex](https://spyndex.readthedocs.io/en/latest/) adossée à la base de donnée d'indices spectraux [Awesome Spectral Indices](https://github.com/davemlz/awesome-ee-spectral-indices) qui offre un gigantesque choix d'indices spectraux.

✅ A noter qu'EOReader calcule automatiquement le nodata et transforme les bandes en réflecctance, permettant ainsi une meilleure comparaison radiométrique entre capteurs.

In [ ]:
# Load du NBR et du MNDWI pré à 30 m
pre_bands = pre.load((NBR, MNDWI), window=aoi, pixel_size=post.pixel_size)
pre_nbr = pre_bands[NBR]
pre_mndwi = pre_bands[MNDWI]

# Load du NBR post
post_nbr = post.load(NBR, window=aoi)[NBR]

Comme vous le voyez, le calcul de ces indices spectraux est indépendant du capteur : 
- il est inutile de connaitre la structure interne du produit
- ni de savoir à quel bande ID correspond quel bande spectrale
- ou bien encore de connaitre les formules des indices spectraux.

In [ ]:
# Analyse de la bande NBR
post_nbr

In [ ]:
# Plot du NBR post
post_nbr.plot(cmap="gist_heat_r", robust=True)

La tache de feux semble être visible dans la bande `NBR` post.

Il est maintenant temps de calculer la différence de `NBR` (`dNBR`) entre l'image pré et post

In [ ]:
# Reprojection de la donnée post sur la grille de l'image pré
post_nbr = post_nbr.rio.reproject_match(pre_nbr)

In [ ]:
dnbr = pre_nbr - post_nbr

In [ ]:
dnbr.plot(robust=True, cmap="gist_heat")

La tache de feu est encore mieux discriminée dans la bande `dNBR`.

Essayons de seuiller les indices spectraux que nous avons sous la main afin d'extraire la tache de feu.  
D'après la figure ci-dessus, un seuil de `dNBR` à 0.25 semble être pertinent.  
Cependant la mer et les étangs sortent aussi en clair, ce qui indique qu'un seuil seulement sur le dNBR risque de faire apparait l'eau de référence comme une zone brulée.

Un seuil usuel pour retirer l'eau de référence est `MNDWI < 0` que nous allons appliquer.

In [ ]:
burnt_area = (dnbr > 0.25) & (pre_mndwi < 0)

Nous allons ensuite comparer cette tache de feu extraite à la va-vite avec le vecteur de vérité terrain tiré de [Copernicus Rapid Mapping (EMSR899)](https://mapping.emergency.copernicus.eu/activations/EMSR899/aois) afin de comparer.

In [ ]:
truth = gpd.read_file("EMSR899_AOI01_GRA_MONIT01_observedEventA_v2.geojson")

plt.figure(figsize=(16, 8))
ax = plt.subplot()
burnt_area.plot(ax=ax, cmap="Reds", label="Zone brulée")
truth.to_crs(burnt_area.rio.crs).boundary.plot(ax=ax, label="Vérité terrain (CEMS RM)", color="orange")
plt.legend()

✅ La tache de feu extraite sur la base de deux capteurs différents est bien cohérente avec la vérité terrain.

## Deuxième partie : EOSets

Nous allons réitérer le processus en utilisant la bibliothèque [EOSets](https://eosets.readthedocs.io/latest/).  
Alors qu'EOReader se concentre sur la gestion d'une seule image satellite, EOSets a été conçue pour gérer les interactiions entre plusieurs produits EOReader (mosaiques, paires, séries).

<img src="https://eosets.readthedocs.io/latest/docs/_static/eosets_objects.png" alt="EOSets" width="600">

Comme nous le voyons, l'objet de base d'EOSets est la mosaique.  
Une série est composée d'une liste de mosaiques avec une référence à choisir.  

Une paire est composée de deux mosaiques qui se superposent.  
Cet objet permet de loader des bandes de différence entre mosaique de référence et secondaire.  
C'est cet objet que nous allons utiliser pour extraire du feu.  


In [ ]:
# New imports
from eosets import Pair

Pour créer une paire dans EOSets, nous n'avons besoin que des chemins des données.


Etant données que la composante de référence et la composante secondaire peuvent être des mosaique, il est possible de donner une liste de chemin pour ces deux variables.  
Les données seront mosaiquées avant d'être interprétée.

In [ ]:
# Create the pair
pair = Pair(
    reference_paths=s2_pre_path, 
    secondary_paths=l9_post_path
)

Comme précédemment, nous allons loader une bande `dNBR` et une bande `MNDWI` pré.

In [ ]:
# Load the dNBR
pre_ds, _, dnbr_ds = pair.load(
    reference_bands=MNDWI,
    diff_bands=NBR,
    window=aoi_path,
)
pre_mndwi = pre_ds[MNDWI]
dnbr = dnbr_ds[NBR]

In [ ]:
# Plot the dNBR
dnbr.plot(
    robust=True,
    cmap="gist_heat",
)
plt.show()

Appliquons ensuite le même traitement : 

In [ ]:
burnt_area = (dnbr > 0.25) & (pre_mndwi < 0)

Puis vérifions si le résultat est cohérent : 

In [ ]:
plt.figure(figsize=(16, 8))
ax = plt.subplot()
burnt_area.plot(ax=ax, cmap="Reds", label="Zone brulée")
truth.to_crs(burnt_area.rio.crs).boundary.plot(ax=ax, label="Vérité terrain (CEMS RM)", color="orange")
plt.legend()